In [1]:
import json
import os
import  numpy as np
import pandas as pd
from keras_preprocessing.image import img_to_array
from multiprocess.pool import worker
from tensorflow.keras import Model
from tensorflow.keras.utils import Sequence,load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten

In [2]:
test_img="./arcade/stenosis/test/images/"
train_img="./arcade/stenosis/train/images/"
val_img="./arcade/stenosis/val/images/"

In [3]:
js_train="./arcade/stenosis/train/annotations/train.json"
js_val="./arcade/stenosis/val/annotations/val.json"
js_test="./arcade/stenosis/test/annotations/test.json"

with open (js_train,"r") as f:
    js_tra=json.load(f)

with open (js_val,"r") as b:
    js_v=json.load(b)

with open (js_test,"r") as c:
    js_te=json.load(c)

In [4]:
print(f" count train:{len(os.listdir(train_img))}")
print(f" count val:{len(os.listdir(val_img))}")
print(f" count test:{len(os.listdir(test_img))}")

 count train:1000
 count val:200
 count test:300


In [5]:
import shutil
import os
import json
from pathlib import Path

if os.path.exists("./yolo_dataset/labels"):
    shutil.rmtree("./yolo_dataset/labels")
    print("labels folder deleted")

BASE_DIR="./arcade/stenosis"
OUTPUT_DIR="./yolo_dataset"

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT_DIR}/labels/{split}', exist_ok=True)

for split in ['train', 'val', 'test']:
    json_path=f"{BASE_DIR}/{split}/annotations/{split}.json"

    print(f"\nProcessing {split}...")

    with open(json_path, 'r') as f:
        data=json.load(f)

    image_dict={img['id']: img for img in data['images']}

    annotations_by_image={}
    for ann in data['annotations']:
        img_id=ann['image_id']
        if img_id not in annotations_by_image:
            annotations_by_image[img_id]=[]
        annotations_by_image[img_id].append(ann)

    for img_id, annotations in annotations_by_image.items():
        img_info=image_dict[img_id]
        file_name=img_info['file_name']
        width=img_info['width']
        height=img_info['height']

        label_file=f"{OUTPUT_DIR}/labels/{split}/{Path(file_name).stem}.txt"
        with open(label_file, 'w') as f:
            for ann in annotations:
                category_id=ann['category_id'] - 1
                segmentation=ann['segmentation'][0]
                seg_normalized=[]
                for i in range(0, len(segmentation), 2):
                    x=segmentation[i]/width
                    y=segmentation[i+1]/height
                    seg_normalized.extend([x, y])
                seg_str=' '.join([f"{coord:.6f}" for coord in seg_normalized])
                f.write(f"{category_id} {seg_str}\n")

    print(f"{split} done")

print("\n Conversion completed!")


Processing train...
train done

Processing val...
val done

Processing test...
test done

 Conversion completed!


In [ ]:
from ultralytics import YOLO
model=YOLO("yolov8n-seg.pt")

In [ ]:
print(type(train_img))

In [ ]:
import os

print(len(os.listdir("./yolo_dataset/images/train")))
print(len(os.listdir("./yolo_dataset/labels/train")))

In [ ]:
model.train(data="dataset.yaml",epochs=100,batch=8,save=True,imgsz=416,workers=2)
history=model.val()